In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("m2_swarm.ipynb")

# M2 — The swarm examined, and construction begins

**TC6035 Part 1 · Milestone 2 · due after Session 3**

Three things that separate reporting a result from defending one: attributing a
difference to a cause, locating your parameters in theory, and diagnosing
stagnation with a measure rather than by eye.

> **Must cite M1**: your tuned tabu configuration and your initial PSO
> parameters. Not scored otherwise.

> **Carry forward to M3**: your best PSO configuration **and the evidence that
> selected it**.

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

from tc6035 import build_instance, SensorPlacementProblem
from tc6035.runlog import RunSet, save_runset, load_runset
from solutions import ALGORITHMS

STUDENT_ID = "A01234567"   # <-- yours
SALT = 0
BUDGET, N_SEEDS = 5_000, 30
instance = build_instance(STUDENT_ID, salt=SALT)

# --- carried forward from M1 -------------------------------------------------
TABU_TENURE   = 12                    # M1 Question 2
PSO_W, PSO_C1, PSO_C2, PSO_SWARM = 0.7298, 1.4962, 1.4962, 20   # M1 Question 3

print(f"M1 carried forward: tenure {TABU_TENURE}, "
      f"(w,c1,c2)=({PSO_W},{PSO_C1},{PSO_C2}), swarm {PSO_SWARM}")

---
## Question 1 — Topology, from the same seeds

Run PSO with `topology="gbest"` and `topology="ring"` over the **same**
`N_SEEDS` seeds. Pairing matters: it removes seed variance from the comparison,
so what remains is attributable to the topology.

Store `p_topology` and save both run logs.

In [ ]:
topo_scores = {}
for topo in ("gbest", "ring"):
    recs, vals = [], []
    for seed in range(N_SEEDS):
        prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
        vals.append(ALGORITHMS["pso"](prob, seed, topology=topo,
                                      swarm_size= ...
        recs.append(prob.record)
    topo_scores[topo] = np.array(vals)
    save_runset(f"results/m2_pso_{topo}.npz", RunSet(algorithm=f"pso_{topo}",
                student_id=STUDENT_ID, params={"topology": topo}, records=recs))

W_topo, p_topology = wilcoxon(topo_scores["gbest"], topo_scores["ring"],
                              alternative= ...

for t, v in topo_scores.items():
    print(f"  {t:6} median {np.median(v):.4f}  IQR [{np.percentile(v,25):.4f}, {np.percentile(v,75):.4f}]")
print(f"\n  gbest wins {int((topo_scores['gbest'] > topo_scores['ring']).sum())}/{N_SEEDS}")
print(f"  median paired difference {np.median(topo_scores['gbest'] - topo_scores['ring']):+.4f}")
print(f"  Wilcoxon W={W_topo:.1f}  p={p_topology:.3e}")

In [ ]:
grader.check("q1_topology")

---
## Question 2 — Locate yourself in the stability region

The deterministic recurrence $x_{k+1} = (1+w-\varphi)x_k - w x_{k-1}$ converges
when $w < 1$ and $0 < \varphi < 2(1+w)$, with $\varphi = c_1 + c_2$.

Compute `phi`, `stability_bound` and `inside_region` for **your** parameters —
and then *verify it* by iterating the recurrence rather than trusting the
inequality.

In [ ]:
phi = ...
stability_bound = ...
inside_region = ...

def recurrence_final(w, phi, steps=200):
    """|displacement| after `steps`, from the deterministic recurrence."""
    x_prev, x = ...
    ...
        x_prev, x = ...
        ...
            ...
    ...

final = recurrence_final(PSO_W, phi)
converged = np.isfinite(final) and final < 1e-3

print(f"phi = {phi:.4f}   bound 2(1+w) = {stability_bound:.4f}")
print(f"predicted: {'inside' if inside_region else 'outside'}")
print(f"simulated: |x| after 200 steps = {final:.3e}  ->  {'converges' if converged else 'diverges'}")
print(f"agreement: {inside_region == converged}")

In [ ]:
grader.check("q2_stability")

---
## Question 3 — Did your runs stagnate? Decide with a measure

A flat convergence curve is ambiguous: it looks the same whether the swarm found
the optimum or merely agreed with itself. So do not eyeball it — measure.

From your gbest run logs, compute for each seed the fraction of the budget at
which the best-so-far **last improved**, and store the worst case in
`earliest_stall`. Then produce the figure: best-so-far and a dispersion proxy on
the same time axis, saved to `figures/`.

> **Either answer is correct, and you must say which you got.** If the last
> improvement lands near the end of the budget, your runs did *not* stagnate —
> the budget is your binding constraint, not diversity, and that changes what
> you would do with more resources. If it lands early, you have premature
> convergence and the dispersion proxy should corroborate it. Reporting
> stagnation you did not observe is worse than reporting none.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

runs = load_runset("results/m2_pso_gbest.npz")

stall_fractions = ...
...
    gains = ...
    ...
stall_fractions = ...
earliest_stall = ...
worst_seed = ...

Path("figures").mkdir(exist_ok=True)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
best = runs.best_so_far[worst_seed]
ax1.plot(best, lw=2)
ax1.axvline(earliest_stall * best.size, ls="--", c="crimson")
ax1.set_ylabel("best so far")
ax1.set_title(f"seed {runs.seeds[worst_seed]} — last improvement at "
              f"{earliest_stall:.0%} of budget")
# A dispersion proxy: spread of evaluated values in a sliding window. Collapsing
# spread with a flat best-so-far is the signature of a swarm that agreed with
# itself rather than one that solved the problem.
vals = runs.values[worst_seed]
window = 100
spread = np.array([vals[i:i+window].std() for i in range(0, vals.size - window, 10)])
ax2.plot(np.arange(spread.size) * 10, spread, lw=2, c="darkorange")
ax2.set_ylabel("evaluation spread"); ax2.set_xlabel("evaluations")
fig.tight_layout(); fig.savefig("figures/m2_stagnation.png", dpi=110)

print(f"last improvement, median over seeds: {np.median(stall_fractions):.1%} of budget")
print(f"earliest stall: {earliest_stall:.1%} (seed {runs.seeds[worst_seed]})")
print(f"budget spent after the last gain, worst seed: {(1-earliest_stall)*BUDGET:.0f} evaluations")

In [ ]:
grader.check("q3_stagnation")

---
## Question 4 — ACO as subset construction

Implement `aco(problem, seed, **params)` in `src/solutions.py`.

This is **not** a tour. Pheromone sits on candidate sites; construction adds one
site at a time; the heuristic is **marginal** coverage given what the ant has
already placed. A static heuristic lets every ant pile onto the same sites.

Use MMAS-style bounds and deposit from the **iteration** best, not the global
best. Run over `N_SEEDS` seeds and save the log.

In [ ]:
ACO_ALPHA, ACO_BETA, ACO_RHO, ACO_ANTS = ...

aco_records, aco_scores = [], []
for seed in range(N_SEEDS):
    prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
    aco_scores.append(ALGORITHMS["aco"](prob, seed, alpha=ACO_ALPHA, beta=ACO_BETA,
                                        rho= ...
    aco_records.append(prob.record)
aco_scores = np.array(aco_scores)

save_runset("results/m2_aco.npz", RunSet(algorithm="aco", student_id=STUDENT_ID,
            params={"alpha": ACO_ALPHA, "beta": ACO_BETA, "rho": ACO_RHO,
                    "n_ants": ACO_ANTS}, records=aco_records))

print(f"ACO median {np.median(aco_scores):.4f}  "
      f"IQR [{np.percentile(aco_scores,25):.4f}, {np.percentile(aco_scores,75):.4f}]")

In [ ]:
grader.check("q4_aco")

<!-- BEGIN QUESTION -->

---
## Question 5 — Analysis

350–500 words. All four:

1. **Topology.** What does your test license you to claim? Check whether your
   median and your win-count agree — if they disagree, say what that means.
2. **Stability.** Where are your parameters, and what does that guarantee?
   State precisely what it does **not** guarantee.
3. **Stagnation.** What fraction of your budget was spent after the last
   improvement, and what would you change?
4. **ACO.** Is your bound actually binding? How would you show it — and what
   would you expect to see if you removed it?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Before you submit

- [ ] `ALGORITHMS` now includes `aco`
- [ ] `results/` holds `m2_pso_gbest.npz`, `m2_pso_ring.npz`, `m2_aco.npz`
- [ ] `figures/m2_stagnation.png` comes from **your** run logs
- [ ] This notebook cites **M1** by name
- [ ] Carry forward to M3: best PSO configuration *and the evidence*
- [ ] `python scripts/self_check.py` passes